# DeepShield V5 -- Kaggle GPU Training Pipeline
**Status**: PRE-TRAINING SETUP & VERIFICATION (TRAINING NOT STARTED)

This notebook prepares and verifies the DeepShield V5 training pipeline for execution on Kaggle GPU.

### Architecture & Pipeline Overview:
- **Backbone**: EfficientNetB2 (ImageNet pretrained, frozen in Stage 1)
- **Input Dimensions**: 224 x 224 x 3 (RGB)
- **Forensic Classification Head**: GlobalAveragePooling2D -> Dropout(0.3) -> Dense(256) -> Dropout(0.3) -> Dense(1, sigmoid)
- **Class Mapping**: `0 = Real`, `1 = Fake` (Higher sigmoid score = Fake)
- **Training Strategy**: Two-Stage Transfer Learning
  - Stage 1: Frozen EfficientNetB2 backbone (15 epochs, learning rate = 1e-3, Adam)
  - Stage 2: Fine-tuning upper ~30% layers with BatchNormalization layers kept frozen (25 epochs, learning rate = 1e-5, Adam)
- **Dataset**: `Face_Cropped_FAST10` (24,589 total images: 17,208 train, 3,688 validation, 3,693 test)
- **Manifest**: `face_crop_FAST10_manifest_split.csv`
- **Target Output**: `/kaggle/working/DeepShield_V5/` (Checkpoints, Model, Histories, Evaluation)
- **Environment**: Kaggle GPU (Tesla T4 x2 or P100)


In [ ]:
# CELL 2 -- TENSORFLOW / PACKAGE VERIFICATION
import sys
import os
import platform

print("=" * 60)
print("DEEPSHIELD V5: PACKAGE & RUNTIME VERIFICATION")
print("=" * 60)
print(f"Python Version:     {sys.version.split()[0]} ({platform.platform()})")

try:
    import tensorflow as tf
    print(f"TensorFlow Version: {tf.__version__}")
    tf_ok = True
except ImportError as e:
    print(f"[!] TensorFlow Import Error: {e}")
    tf_ok = False

try:
    import numpy as np
    print(f"NumPy Version:      {np.__version__}")
except ImportError:
    pass

try:
    import pandas as pd
    print(f"Pandas Version:     {pd.__version__}")
except ImportError:
    pass

try:
    import sklearn
    print(f"Scikit-Learn:       {sklearn.__version__}")
except ImportError:
    pass

print("=" * 60)
if tf_ok:
    print("PACKAGE CHECK: PASS")
else:
    print("PACKAGE CHECK: FAILED")
print("=" * 60)


In [ ]:
# CELL 3 -- KAGGLE GPU VERIFICATION
import tensorflow as tf

print("=" * 60)
print("KAGGLE GPU HARDWARE CHECK")
print("=" * 60)

print(f"TensorFlow version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
gpu_available = len(gpus) > 0

print(f"GPU available:      {gpu_available}")
print(f"Number of GPUs:     {len(gpus)}")

if gpu_available:
    for i, gpu in enumerate(gpus):
        try:
            details = tf.config.experimental.get_device_details(gpu)
            device_name = details.get('device_name', gpu.name)
        except Exception:
            device_name = gpu.name
        print(f"GPU device [{i}]:     {device_name}")
    print("=" * 60)
    print("GPU STATUS: READY (Kaggle GPU accelerator detected)")
else:
    print("GPU device name:    None")
    print("=" * 60)
    print("[!] GPU NOT DETECTED!")
    print("[!] Please enable GPU in Kaggle settings:")
    print("    Notebook Settings (right panel) -> Accelerator -> Select 'GPU T4 x2' or 'GPU P100'")
print("=" * 60)


In [ ]:
# CELL 4 -- LOCATE KAGGLE DATASET AND MANIFEST
import os
from pathlib import Path

# CONFIGURABLE PATHS:
# If you attached your dataset under a custom Kaggle name, specify paths below or leave blank to auto-detect:
KAGGLE_DATASET_ROOT = "/kaggle/input/datasets/devyaninandankar/deepshield-face-dataset/Kaggle_Transfer/Face_Cropped_FAST10"
KAGGLE_MANIFEST_PATH = "/kaggle/input/datasets/devyaninandankar/deepshield-face-dataset/Kaggle_Transfer/face_crop_FAST10_manifest_split.csv"

print("=" * 70)
print("KAGGLE INPUT DATASET SCANNER (/kaggle/input)")
print("=" * 70)

input_base = Path("/kaggle/input")
matching_dataset_dirs = []
matching_manifest_files = []
auto_dataset = None
auto_manifest = None

if input_base.exists():
    for root, dirs, files in os.walk(str(input_base)):
        for d in dirs:
            p = Path(root) / d
            if d == "Face_Cropped_FAST10" or "fast10" in d.lower():
                matching_dataset_dirs.append(p)
                if not auto_dataset and (p / "train").exists():
                    auto_dataset = p
        for f in files:
            p = Path(root) / f
            if "manifest" in f.lower() and f.endswith(".csv"):
                matching_manifest_files.append(p)
                if not auto_manifest and "split" in f.lower():
                    auto_manifest = p

print("Attached dataset directories found:")
if matching_dataset_dirs:
    for d in matching_dataset_dirs:
        print(f"  - {d}")
else:
    print("  [None found in /kaggle/input]")

print("\nAttached manifest files found:")
if matching_manifest_files:
    for f in matching_manifest_files:
        print(f"  - {f}")
else:
    print("  [None found in /kaggle/input]")

# Resolve active dataset and manifest paths
if KAGGLE_DATASET_ROOT and Path(KAGGLE_DATASET_ROOT).is_dir():
    ACTIVE_DATASET_ROOT = Path(KAGGLE_DATASET_ROOT)
    print(f"\n[+] Using user-configured KAGGLE_DATASET_ROOT: {ACTIVE_DATASET_ROOT}")
elif auto_dataset:
    ACTIVE_DATASET_ROOT = auto_dataset
    print(f"\n[+] Using auto-detected dataset directory: {ACTIVE_DATASET_ROOT}")
else:
    ACTIVE_DATASET_ROOT = None
    print("\n[!] DATASET ROOT NOT FOUND: Please click '+ Add Input' in Kaggle and attach Face_Cropped_FAST10.")

if KAGGLE_MANIFEST_PATH and Path(KAGGLE_MANIFEST_PATH).is_file():
    ACTIVE_MANIFEST_PATH = Path(KAGGLE_MANIFEST_PATH)
    print(f"[+] Using user-configured KAGGLE_MANIFEST_PATH: {ACTIVE_MANIFEST_PATH}")
elif auto_manifest:
    ACTIVE_MANIFEST_PATH = auto_manifest
    print(f"[+] Using auto-detected manifest CSV: {ACTIVE_MANIFEST_PATH}")
else:
    ACTIVE_MANIFEST_PATH = None
    print("[!] MANIFEST NOT FOUND: Please click '+ Add Input' in Kaggle and attach the manifest CSV.")

print("=" * 70)


In [ ]:
# CELL 5 -- VERIFY DATASET COUNTS
import os
from pathlib import Path

EXPECTED_COUNTS = {
    "train": 17208,
    "validation": 3688,
    "test": 3693,
    "total": 24589
}

print("=" * 60)
print("DEEPSHIELD V5: KAGGLE DATASET IMAGE COUNT AUDIT")
print("=" * 60)

valid_exts = {".jpg", ".jpeg", ".png"}
actual_counts = {}

if ACTIVE_DATASET_ROOT and ACTIVE_DATASET_ROOT.is_dir():
    for split in ["train", "validation", "test"]:
        s_dir = ACTIVE_DATASET_ROOT / split
        if s_dir.is_dir():
            count = sum(1 for f in os.scandir(s_dir) if f.is_file() and Path(f.name).suffix.lower() in valid_exts)
        else:
            count = 0
        actual_counts[split] = count

    actual_total = sum(actual_counts.values())
    actual_counts["total"] = actual_total

    print(f"Dataset Root:       {ACTIVE_DATASET_ROOT}")
    print(f"Train images:       {actual_counts['train']:,}  (Expected: {EXPECTED_COUNTS['train']:,})")
    print(f"Validation images:  {actual_counts['validation']:,}  (Expected: {EXPECTED_COUNTS['validation']:,})")
    print(f"Test images:        {actual_counts['test']:,}  (Expected: {EXPECTED_COUNTS['test']:,})")
    print(f"Total images:       {actual_counts['total']:,}  (Expected: {EXPECTED_COUNTS['total']:,})")
    print("=" * 60)

    dataset_pass = (
        actual_counts['train'] == EXPECTED_COUNTS['train'] and
        actual_counts['validation'] == EXPECTED_COUNTS['validation'] and
        actual_counts['test'] == EXPECTED_COUNTS['test'] and
        actual_counts['total'] == EXPECTED_COUNTS['total']
    )
    if dataset_pass:
        print("DATASET COUNT AUDIT: PASS (Exact 24,589 images verified)")
    else:
        print("DATASET COUNT AUDIT: MISMATCH DETECTED")
else:
    dataset_pass = False
    print("[!] ACTIVE_DATASET_ROOT is not configured or does not exist.")
    print("DATASET COUNT AUDIT: FAILED")
print("=" * 60)


In [ ]:
# CELL 6 -- VERIFY MANIFEST AND CLASS MAPPING
from pathlib import Path
import pandas as pd

EXPECTED_RECORDS = 24589
EXPECTED_COLUMNS = {"label", "split", "video_group"}

print("=" * 60)
print("DEEPSHIELD V5: KAGGLE MANIFEST & CLASS MAPPING AUDIT")
print("=" * 60)

manifest_pass = False
if ACTIVE_MANIFEST_PATH and ACTIVE_MANIFEST_PATH.is_file():
    df = pd.read_csv(ACTIVE_MANIFEST_PATH)
    total_records = len(df)
    missing_cols = EXPECTED_COLUMNS - set(df.columns)

    print(f"Manifest Path:      {ACTIVE_MANIFEST_PATH}")
    print(f"Total Records:      {total_records:,} (Expected: {EXPECTED_RECORDS:,})")
    print(f"Required Columns:   {'PASS' if not missing_cols else f'MISSING: {missing_cols}'}")

    # Class mapping check
    unique_labels = sorted(df['label'].dropna().unique())
    print("\nClass Mapping Audit:")
    print("  0 = Real")
    print("  1 = Fake (Higher sigmoid score = Fake)")
    print(f"  Observed Labels:  {unique_labels}")

    real_count = int((df['label'] == 0).sum())
    fake_count = int((df['label'] == 1).sum())
    print(f"  Real (0) samples: {real_count:,}")
    print(f"  Fake (1) samples: {fake_count:,}")

    manifest_pass = (total_records == EXPECTED_RECORDS and not missing_cols and unique_labels == [0, 1])
    print("=" * 60)
    if manifest_pass:
        print("MANIFEST AUDIT: PASS (24,589 records & verified 0/1 class mapping)")
    else:
        print("MANIFEST AUDIT: CHECK REQUIRED")
else:
    print("[!] ACTIVE_MANIFEST_PATH not configured or file not found.")
    print("MANIFEST AUDIT: FAILED")
print("=" * 60)


In [ ]:
# CELL 7 -- VERIFY TRAIN/VALIDATION/TEST SPLIT INTEGRITY
import pandas as pd

EXPECTED_SPLITS = {
    "train": 17208,
    "validation": 3688,
    "test": 3693
}

print("=" * 60)
print("DEEPSHIELD V5: SPLIT INTEGRITY & GROUP LEAKAGE AUDIT")
print("=" * 60)

split_pass = False
if ACTIVE_MANIFEST_PATH and ACTIVE_MANIFEST_PATH.is_file():
    df = pd.read_csv(ACTIVE_MANIFEST_PATH)

    # 1. Split count distribution
    split_counts = df['split'].value_counts().to_dict()
    print("Observed Split Counts:")
    counts_match = True
    for s_name, exp_num in EXPECTED_SPLITS.items():
        act_num = split_counts.get(s_name, 0)
        match_flag = "PASS" if act_num == exp_num else "MISMATCH"
        print(f"  - {s_name.capitalize():<12}: {act_num:>6,} (Expected: {exp_num:>6,}) [{match_flag}]")
        if act_num != exp_num:
            counts_match = False

    # 2. Video group leakage audit
    train_groups = set(df[df['split'] == 'train']['video_group'].dropna().unique())
    val_groups = set(df[df['split'] == 'validation']['video_group'].dropna().unique())
    test_groups = set(df[df['split'] == 'test']['video_group'].dropna().unique())

    leakage_train_val = train_groups & val_groups
    leakage_train_test = train_groups & test_groups
    leakage_val_test = val_groups & test_groups

    total_leakage = len(leakage_train_val) + len(leakage_train_test) + len(leakage_val_test)

    print("\nVideo Group Leakage Check:")
    print(f"  - Unique Train groups:      {len(train_groups):,}")
    print(f"  - Unique Validation groups: {len(val_groups):,}")
    print(f"  - Unique Test groups:       {len(test_groups):,}")
    print(f"  - Train/Val Leakage:        {len(leakage_train_val)} overlapping groups")
    print(f"  - Train/Test Leakage:       {len(leakage_train_test)} overlapping groups")
    print(f"  - Val/Test Leakage:         {len(leakage_val_test)} overlapping groups")

    split_pass = counts_match and (total_leakage == 0)
    print("=" * 60)
    if split_pass:
        print("SPLIT & LEAKAGE AUDIT: PASS (Zero group leakage across splits)")
    else:
        print("SPLIT & LEAKAGE AUDIT: FAILED")
else:
    print("[!] Manifest not available for split integrity check.")
    print("SPLIT & LEAKAGE AUDIT: FAILED")
print("=" * 60)


In [ ]:
# SETUP CELL -- UPLOAD & CONFIGURE TRAINING SCRIPT
# Guarantees that /kaggle/working/train_deepshield_v5.py has the latest 5.0.3-face-path-fix script.
import os
import sys
import shutil
import base64
from pathlib import Path

target_script = Path("/kaggle/working/train_deepshield_v5.py")
EXPECTED_VERSION = "5.0.3-face-path-fix"

print("=" * 70)
print("DEEPSHIELD V5: TRAINING SCRIPT SYNCHRONIZATION")
print("=" * 70)

# 1. First priority: check attached Kaggle datasets in /kaggle/input
CANDIDATE_DIRS = [
    Path("/kaggle/input/datasets/devyaninandankar/deepshield-v5-code"),
    Path("/kaggle/input/deepshield-v5-code"),
    Path("/kaggle/input/deepshield-training-script"),
    Path("/kaggle/input/face_cropped_fast10"),
    Path("/kaggle/input")
]

best_source = None
for cdir in CANDIDATE_DIRS:
    if cdir.exists():
        for root, dirs, files in os.walk(str(cdir)):
            if "train_deepshield_v5.py" in files:
                p = Path(root) / "train_deepshield_v5.py"
                try:
                    txt = p.read_text(encoding="utf-8", errors="ignore")
                    if EXPECTED_VERSION in txt:
                        best_source = p
                        break
                except Exception:
                    pass
        if best_source:
            break

if best_source:
    shutil.copy2(best_source, target_script)
    source_desc = f"Kaggle dataset: {best_source}"
else:
    # 2. Embedded latest 5.0.3-face-path-fix script (guarantees update even without dataset upload)
    EMBEDDED_B64 = "..."
    decoded_bytes = base64.b64decode(EMBEDDED_B64)
    target_script.write_bytes(decoded_bytes)
    source_desc = "Synchronized from latest verified 5.0.3-face-path-fix build"

# 3. Verify target script on disk
file_size = target_script.stat().st_size
script_content = target_script.read_text(encoding="utf-8", errors="ignore")

found_version = None
for line in script_content.splitlines():
    if "SCRIPT_VERSION" in line and "=" in line:
        found_version = line.split("=")[-1].strip().strip('"').strip("'")
        break

print(f"Source:            {source_desc}")
print(f"Destination path:  {target_script}")
print(f"File size:         {file_size:,} bytes")
print(f"SCRIPT_VERSION:    {found_version}")
print("=" * 70)

# Hard Check
if found_version != EXPECTED_VERSION or "face_path" not in script_content:
    print("[!] CRITICAL ERROR: STALE SCRIPT DETECTED!")
    sys.exit("STALE TRAINING SCRIPT DETECTED — DO NOT TRAIN")

print("[+] SCRIPT INTEGRITY CHECK: PASS")
print("TRAINING SCRIPT READY")
print("=" * 70)


In [ ]:
# CELL 8 -- VERIFY TRAIN_DEEPSHIELD_V5.PY
import os
import sys
import py_compile
from pathlib import Path

print("=" * 60)
print("DEEPSHIELD V5: TRAINING SCRIPT INTEGRITY AUDIT")
print("=" * 60)

script_path = Path("/kaggle/working/train_deepshield_v5.py")

if not script_path.is_file():
    print("[!] Training script not found at /kaggle/working/train_deepshield_v5.py")
    sys.exit(1)

# 1. Python Syntax Check
try:
    py_compile.compile(str(script_path), doraise=True)
    syntax_pass = True
    print("Syntax:                  PASS")
except Exception as exc:
    syntax_pass = False
    print(f"Syntax:                  FAIL ({exc})")

with open(script_path, "r", encoding="utf-8") as f:
    content = f.read()

# 2. Version Verification
has_version = 'SCRIPT_VERSION = "5.0.3-face-path-fix"' in content
print(f"Script version:          {'5.0.3-face-path-fix (PASS)' if has_version else 'FAIL'}")

# 3. face_path Resolver Verification
has_face_path_resolver = "face_path" in content and "bname = Path(str(row['face_path'])" in content.replace('"', "'")
if not has_face_path_resolver:
    has_face_path_resolver = "face_path" in content and "resolve_split_filepaths" in content
print(f"face_path resolver:      {'PASS' if has_face_path_resolver else 'FAIL'}")

# 4. EfficientNetB2 Backbone
has_effnet = "EfficientNetB2" in content
print(f"EfficientNetB2:          {'PASS' if has_effnet else 'FAIL'}")

# 5. Input Dimension 224x224x3
has_input_size = "IMAGE_SIZE = (224, 224)" in content
print(f"Input 224x224x3:         {'PASS' if has_input_size else 'FAIL'}")

# 6. BatchNorm Frozen
has_freeze_bn = "isinstance(layer, layers.BatchNormalization)" in content
print(f"BatchNorm frozen:        {'PASS' if has_freeze_bn else 'FAIL'}")

script_pass = syntax_pass and has_version and has_face_path_resolver and has_effnet and has_input_size and has_freeze_bn

print("=" * 60)
if script_pass:
    print("TRAINING SCRIPT AUDIT: PASS")
else:
    print("TRAINING SCRIPT AUDIT: FAIL — STALE OR INVALID SCRIPT")
    sys.exit("STALE TRAINING SCRIPT DETECTED — DO NOT TRAIN")
print("=" * 60)


In [ ]:
# CELL 9 -- KAGGLE PATH & ENVIRONMENT CONFIGURATION
import os
import sys
import shutil
import pandas as pd
from pathlib import Path

# 1. Set runtime environment variables for training script
os.environ["ENVIRONMENT"] = "kaggle"
DATASET_DIR = Path("/kaggle/input/datasets/devyaninandankar/deepshield-face-dataset/Kaggle_Transfer/Face_Cropped_FAST10")
MANIFEST_FILE = Path("/kaggle/input/datasets/devyaninandankar/deepshield-face-dataset/Kaggle_Transfer/face_crop_FAST10_manifest_split.csv")

# Fallback to active variables if mounted at alternate path
if not DATASET_DIR.is_dir() and 'ACTIVE_DATASET_ROOT' in globals() and ACTIVE_DATASET_ROOT:
    DATASET_DIR = Path(ACTIVE_DATASET_ROOT)
if not MANIFEST_FILE.is_file() and 'ACTIVE_MANIFEST_PATH' in globals() and ACTIVE_MANIFEST_PATH:
    MANIFEST_FILE = Path(ACTIVE_MANIFEST_PATH)

os.environ["KAGGLE_DATASET_ROOT"] = str(DATASET_DIR)
os.environ["KAGGLE_MANIFEST_PATH"] = str(MANIFEST_FILE)

# 2. Establish required output directory hierarchy in /kaggle/working/
KAGGLE_OUTPUT_ROOT = Path("/kaggle/working/DeepShield_V5")
KAGGLE_CHECKPOINTS = KAGGLE_OUTPUT_ROOT / "checkpoints"
KAGGLE_TRAINING_LOGS = KAGGLE_OUTPUT_ROOT / "training"
KAGGLE_EVALUATION = KAGGLE_OUTPUT_ROOT / "evaluation"

for folder in [KAGGLE_OUTPUT_ROOT, KAGGLE_CHECKPOINTS, KAGGLE_TRAINING_LOGS, KAGGLE_EVALUATION]:
    folder.mkdir(parents=True, exist_ok=True)

# 3. REAL FILESYSTEM AUDIT USING face_path
print("=" * 60)
print("DEEPSHIELD V5: PATH RESOLUTION AUDIT")
print("=" * 60)
print("\nPath source: face_path\n")

if not DATASET_DIR.is_dir():
    print(f"[!] Dataset directory not found: {DATASET_DIR}")
    sys.exit(1)

if not MANIFEST_FILE.is_file():
    print(f"[!] Manifest CSV not found: {MANIFEST_FILE}")
    sys.exit(1)

manifest_df = pd.read_csv(MANIFEST_FILE)
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SPLIT_EXPECTED = {
    "train": 17208,
    "validation": 3688,
    "test": 3693
}

split_results = {}
first_20_missing = []

for split_name in ["train", "validation", "test"]:
    split_dir = DATASET_DIR / split_name
    if not split_dir.is_dir():
        for cand in [split_name.lower(), split_name.capitalize(), split_name.upper()]:
            if (DATASET_DIR / cand).is_dir():
                split_dir = DATASET_DIR / cand
                break

    # Real recursive scan of actual files on disk
    actual_files_index = {}
    actual_stems_index = {}
    if split_dir.is_dir():
        for root, dirs, files in os.walk(str(split_dir)):
            for f in files:
                if Path(f).suffix.lower() in valid_exts:
                    full_p = os.path.join(root, f)
                    actual_files_index[f] = full_p
                    actual_files_index[f.lower()] = full_p
                    actual_stems_index[Path(f).stem.lower()] = full_p

    # Compare manifest face_path against actual discovered files on disk
    subset = manifest_df[manifest_df['split'] == split_name]
    resolved_count = 0
    missing_list = []

    for idx, row in subset.iterrows():
        matched = False
        if 'face_path' in row and pd.notna(row['face_path']):
            fp_val = str(row['face_path']).strip()
            bname = Path(fp_val.replace("\\", "/")).name
            p_direct = split_dir / bname
            if p_direct.is_file():
                matched = True
            elif bname in actual_files_index or bname.lower() in actual_files_index:
                matched = True
            elif Path(bname).stem.lower() in actual_stems_index:
                matched = True

        if matched:
            resolved_count += 1
        else:
            cand_bname = Path(str(row.get('face_path', 'N/A')).replace("\\", "/")).name
            missing_list.append(cand_bname)
            if len(first_20_missing) < 20:
                first_20_missing.append(f"[{split_name}] row {idx}: {cand_bname}")

    expected_count = SPLIT_EXPECTED.get(split_name, len(subset))
    missing_count = len(missing_list)
    split_results[split_name] = {
        "expected": expected_count,
        "resolved": resolved_count,
        "missing": missing_count
    }

# Format output exactly as requested
total_expected = sum(r['expected'] for r in split_results.values())
total_resolved = sum(r['resolved'] for r in split_results.values())
total_missing = sum(r['missing'] for r in split_results.values())

print("Train:")
print(f"  Expected: {split_results['train']['expected']:,}")
print(f"  Resolved: {split_results['train']['resolved']:,}")
print(f"  Missing: {split_results['train']['missing']:,}\n")

print("Validation:")
print(f"  Expected: {split_results['validation']['expected']:,}")
print(f"  Resolved: {split_results['validation']['resolved']:,}")
print(f"  Missing: {split_results['validation']['missing']:,}\n")

print("Test:")
print(f"  Expected: {split_results['test']['expected']:,}")
print(f"  Resolved: {split_results['test']['resolved']:,}")
print(f"  Missing: {split_results['test']['missing']:,}\n")

print("Total:")
print(f"  Expected: {total_expected:,}")
print(f"  Resolved: {total_resolved:,}")
print(f"  Missing: {total_missing:,}\n")
print("=" * 60)

if total_missing > 0:
    print(f"[!] CRITICAL ERROR: {total_missing:,} missing file(s) detected during face_path audit!")
    print("First 20 missing face_path basenames:")
    for m in first_20_missing:
        print(f"  - {m}")
    print("\nSTOPPING EXECUTION. Do not proceed to training.")
    resolution_pass = False
    sys.exit(1)
else:
    resolution_pass = True
    print("[+] PATH RESOLUTION AUDIT: PASS")
    print("=" * 60)


In [ ]:
# CELL 10 -- FINAL READINESS REPORT
print("=" * 60)
print("DEEPSHIELD V5: KAGGLE FINAL READINESS REPORT")
print("=" * 60)

checks = [
    ("TensorFlow & Packages", globals().get("tf_ok", False)),
    ("GPU Accelerator", globals().get("gpu_available", False)),
    ("Dataset (24,589 images)", globals().get("dataset_pass", False)),
    ("Manifest (24,589 records)", globals().get("manifest_pass", False)),
    ("Split & Leakage Integrity", globals().get("split_pass", False)),
    ("Training Script Valid", globals().get("script_pass", False)),
    ("Path Resolution (3 splits)", globals().get("resolution_pass", False)),
    ("Output Hierarchy Ready", Path("/kaggle/working/DeepShield_V5").is_dir())
]

all_passed = True
for name, passed in checks:
    status_str = "READY / PASS" if passed else "FAILED / INCOMPLETE"
    print(f"  {name:<28}: {status_str}")
    if not passed:
        all_passed = False

print("=" * 60)
if all_passed:
    print("V5 TRAINING ON KAGGLE: READY TO START")
else:
    print("V5 TRAINING ON KAGGLE: CHECK REQUIRED (Resolve failed items above)")
print("=" * 60)
print("\nTRAINING STATUS: NOT STARTED\n")


In [ ]:
# CELL 11 -- START V5 TRAINING
# Execute the verified two-stage training script on Kaggle GPU
# Stage 1: Frozen backbone (15 epochs, lr=1e-3, Adam)
# Stage 2: Fine-tuning upper ~30% layers (25 epochs, lr=1e-5, Adam)
# Target output: /kaggle/working/DeepShield_V5/
!python /kaggle/working/train_deepshield_v5.py
